In [10]:
import pickle
import numpy as np

from tensorflow.keras.models import load_model

model = load_model("best_model.keras")

print("Model Loaded Successfully")

Model Loaded Successfully


In [27]:
with open("notes.pkl", "rb") as f:
    notes = pickle.load(f)

pitchnames = sorted(set(notes))

n_vocab = len(pitchnames)

print("Vocabulary:", n_vocab)

Vocabulary: 1346


In [12]:
note_to_int = dict(
    (note, number)
    for number, note in enumerate(pitchnames)
)

int_to_note = dict(
    (number, note)
    for number, note in enumerate(pitchnames)
)

print("Mappings Created")

Mappings Created


In [13]:
sequence_length = 100

network_input = []

for i in range(
        len(notes)
        - sequence_length):

    sequence = notes[
        i:i+sequence_length
    ]

    network_input.append(
        [
            note_to_int[n]
            for n in sequence
        ]
    )

print("Patterns:", len(network_input))

Patterns: 65550


In [38]:
import numpy as np

start = np.random.randint(
    0,
    len(network_input)-1
)

pattern = list(network_input[start])

prediction_output = []

print("Seed Generated")

Seed Generated


In [39]:
print(type(pattern))
print(len(pattern))
print(pattern[:10] if len(pattern) > 10 else pattern)

<class 'list'>
100
[158, 720, 830, 956, 955, 829, 1335, 209, 1325, 610]


In [40]:
print(type(pattern))
print(len(pattern))

<class 'list'>
100


In [42]:
prediction_output = []

temperature = 0.8

for _ in range(1200):

    prediction_input = np.reshape(
        pattern,
        (1, 100, 1)
    )

    prediction_input = (
        prediction_input / float(n_vocab)
    )

    prediction = model.predict(
        prediction_input,
        verbose=0
    )[0]

    prediction = np.log(
        prediction + 1e-8
    ) / temperature

    exp_preds = np.exp(prediction)

    prediction = (
        exp_preds /
        np.sum(exp_preds)
    )

    index = np.random.choice(
        len(prediction),
        p=prediction
    )

    result = int_to_note[index]

    prediction_output.append(result)

    pattern.append(index)

    pattern = pattern[1:]

print("Generated Notes:", len(prediction_output))

Generated Notes: 1200


In [43]:
print("Unique generated notes:", len(set(prediction_output)))
print(prediction_output[:20])

Unique generated notes: 99
['C5', 'D4', 'A4', 'F#3', 'F#3', 'G4', 'F#5', 'B4', 'G5', 'B4', 'D3', 'G3', 'E4', 'G3', 'D3', 'D5', 'B3', 'E3', 'G5', 'A4']


In [44]:
from music21 import stream
from music21 import note
from music21 import chord
from music21 import instrument

In [45]:
offset = 0
output_notes = []

for pattern in prediction_output:

    try:

        if "." in pattern:

            notes_in_chord = pattern.split(".")

            chord_notes = []

            for current_note in notes_in_chord:

                new_note = note.Note(
                    int(current_note)
                )

                new_note.offset = offset

                chord_notes.append(
                    new_note
                )

            output_notes.extend(
                chord_notes
            )

        else:

            new_note = note.Note(
                pattern
            )

            new_note.offset = offset

            output_notes.append(
                new_note
            )

        offset += 0.25

    except:
        pass

In [46]:
piano = instrument.Piano()

midi_stream = stream.Stream()

midi_stream.append(piano)

for n in output_notes:
    midi_stream.append(n)

midi_stream.write(
    'midi',
    fp='generated_music.mid'
)

print("Music Generated Successfully")

Music Generated Successfully


In [ ]:
import tkinter as tk
from tkinter import messagebox
import pygame
import os

pygame.init()
pygame.mixer.init()

MIDI_FILE = "generated_music.mid"

def play_music():
    try:
        if os.path.exists(MIDI_FILE):
            pygame.mixer.music.load(MIDI_FILE)
            pygame.mixer.music.play()
            status_label.config(text="Playing Music...")
        else:
            messagebox.showerror(
                "Error",
                "generated_music.mid not found"
            )
    except Exception as e:
        messagebox.showerror(
            "Error",
            str(e)
        )

def stop_music():
    pygame.mixer.music.stop()
    status_label.config(text="Music Stopped")

root = tk.Tk()

root.title("AI Music Generator")
root.geometry("500x300")

title = tk.Label(
    root,
    text="AI Music Generation Using LSTM",
    font=("Arial", 16, "bold")
)
title.pack(pady=20)

play_btn = tk.Button(
    root,
    text="Play Generated Music",
    font=("Arial", 12),
    command=play_music,
    width=20
)
play_btn.pack(pady=10)

stop_btn = tk.Button(
    root,
    text="Stop Music",
    font=("Arial", 12),
    command=stop_music,
    width=20
)
stop_btn.pack(pady=10)

status_label = tk.Label(
    root,
    text="Ready",
    font=("Arial", 10)
)
status_label.pack(pady=20)

root.mainloop()

In [1]:
import os
print(os.path.getsize("generated_music.mid"))

15194
